# Streams and Logical Operations

This chapter explains how to treat the collections introduced in the preceding chapters as **streams** and apply operations such as reductions and filtering.
A stream is a sequence of values of a particular type that may contain duplicates. It is similar to what Python calls an **iterator**.
Streams are used when working with indices over a particular range, taking sums or products, and defining indexed constraints.

This chapter also explains logical operations on Boolean values and streams.

In [1]:
import jijmodeling as jm

:::{admonition} Renamed from “Set” in JijModeling 2.8.0
:class: note

Up to JijModeling 2.7.1, streams were called “sets,” and the explicit conversion function was named `jm.set`. Mathematically, however, a set is unordered and contains no duplicates, so the term “set” was misleading.
Since version 2.8.0, this concept has consistently been called a stream.
`jm.set` remains available as a deprecated alias for {py:func}`~jijmodeling.stream`, and existing comprehensions in the Decorator API continue to work, but calling it emits a `DeprecationWarning`.
:::

## Constructing Streams

JijModeling provides mechanisms and functions for automatically converting values of other types into streams, explicitly constructing streams, and composing existing streams into new ones.

### Automatic Conversion from Existing Types

Values of some types are automatically converted into streams. The following table shows specific examples:

| Expression type | Corresponding stream |
| :-------- | :------- |
| Multidimensional array | A stream that traverses elements in row-major order |
| Dictionary | A stream that traverses the dictionary's values |
| Natural-number expression $N$ containing no decision variables | A stream that traverses $0, 1, \ldots, N-1$ |
| Category label `L` | A stream that traverses all values of `L` supplied at compile time |

### Converting a Multidimensional Array into a Stream of Subarrays with {py:func}`~jijmodeling.rows`

As described above, a multidimensional array is automatically converted into a stream that **traverses each element in row-major order**.
To obtain a stream that instead traverses the inner rows in order, use {py:func}`~jijmodeling.rows`.
More precisely, {py:func}`~jijmodeling.rows` converts a multidimensional array of shape $N \times M_1 \times \cdots \times M_n$ into an array of length $N$ whose elements are subarrays of shape $M_1 \times \ldots \times M_n$. The resulting array is then converted into a stream through the automatic conversion described above.

In [2]:
problem = jm.Problem("Row Sum Example")
N = problem.Length("N")
M = problem.Length("M")
K = problem.Length("K")
x = problem.BinaryVar("x", shape=(N, M, K))
problem.infer(x.rows())

Array[N; Array[M, K; binary!]]

:::{admonition} Change from JijModeling 1: Array Traversal
:class: caution

In JijModeling 1, when a multidimensional array appeared in `belong_to=` or `forall=`, its inner rows were traversed in order.
To obtain this behavior, explicitly convert the array with {py:func}`~jijmodeling.rows`, using `jm.rows(A)` or `A.rows()`.
:::

### Obtaining a Stream of Array Indices with {py:meth}`~jijmodeling.Expression.indices`

{py:meth}`~jijmodeling.Expression.indices` returns a stream corresponding to the set of indices—the domain—of an array.

In [3]:
problem = jm.Problem("Index and Keys Example")
S = problem.Float("S", ndim=2)
problem.infer(S.indices())

Stream[Tuple[S.len_at(0), S.len_at(1)]]

### Explicitly Converting a Dictionary into a Stream

As described above, JijModeling's automatic conversion turns a dictionary expression into a stream over its **values, not its keys**.
This differs from Python's {py:class}`dict`, but is intentional so that dictionaries behave consistently with multidimensional arrays.
For example, if a placeholder or decision variable originally defined as a multidimensional array is changed to a dictionary, code such as {py:meth}`x.sum() <jijmodeling.Expression.sum>` does not need to change.
To traverse key-value pairs or keys, use {py:meth}`~jijmodeling.Expression.items` or {py:meth}`~jijmodeling.Expression.keys`. These methods return streams over key-value pairs and keys, respectively.
To make the default conversion to a stream of values explicit, use {py:meth}`~jijmodeling.Expression.values`.

In [4]:
problem = jm.Problem("Row Sum Example")
L = problem.CategoryLabel("L")
N = problem.Length("N")
M = problem.Length("M")
x = problem.TotalDict("x", dict_keys=L, dtype=float)
problem.infer(x.values())

Stream[float]

In [5]:
problem.infer(x.items())

Stream[Tuple[CategoryLabel("L"), float]]

In [6]:
problem.infer(x.keys())

Stream[CategoryLabel("L")]

The following example defines a dictionary of decision variables with the same domain as a `PartialDict` placeholder:

In [7]:
problem = jm.Problem("Index and Keys Example")
N = problem.Length("N")
L = problem.CategoryLabel("L")
S = problem.PartialDict("S", dtype=float, dict_keys=(N, L))
x = problem.BinaryVar("x", dict_keys=S.keys())
problem

Problem(name="Index and Keys Example", sense=MINIMIZE, objective=0, constraints=[])

### Explicit Conversion to a Stream with {py:func}`~jijmodeling.stream`

Conversion to a stream generally happens automatically, but you can explicitly convert a value with {py:func}`~jijmodeling.stream`.
When using the Decorator API, you can also construct a stream directly by passing a comprehension to {py:func}`jm.stream <jijmodeling.stream>`.
Unlike {py:func}`~jijmodeling.genarray` and {py:func}`~jijmodeling.gendict`, {py:func}`~jijmodeling.stream` supports comprehensions containing any number of `for` and `if` clauses.

In [8]:
@jm.Problem.define("Stream Comprehension Example")
def stream_compr_problem(problem: jm.DecoratedProblem):
    N = problem.Natural()
    L = problem.CategoryLabel()
    x = problem.BinaryVar(dict_keys=(L, N))
    display(jm.stream(i + x[l, i] for l in L for i in N if i % 2 == 0))

Expression(stream(stream(L.flat_map(lambda l: N.map(lambda i: (l, i))).filter(lambda (l, i): i % 2 == 0)).map(lambda (l, i): i + x[l, i])))

### Generating Arithmetic Progressions with {py:func}`~jijmodeling.range`

Since JijModeling 2.3.1, {py:func}`~jijmodeling.range`, corresponding to Python's built-in {py:class}`range() <range>`, has been available for defining streams of arithmetic progressions of integers.
Like Python's {py:class}`range() <range>`, when given one argument it traverses values starting from $0$; when given two arguments, it starts from the first and stops before the second; and when given a third argument, that value is used as the step.

In [9]:
range_problem = jm.Problem("Stream Range Example")
N = range_problem.Natural("N")

display(jm.range(N))  # 0, 1, ..., N-1
display(jm.range(1, N))  # 1, 2, ..., N-1
display(jm.range(1, N, 2))  # 1, 3, 5, ... (less than N)

Expression(range(N))

Expression(range(1, N))

Expression(range(1, N, 2))

## Reductions over Streams: Sums, Products, Maximums, Minimums, and More

Streams become especially powerful when combined with reductions such as sums and products. The following section explains the different notations available for sums and products.

:::{note}
For simplicity, the examples below use {py:func}`jm.sum() <jijmodeling.sum>` (or {py:meth}`Expression.sum() <jijmodeling.Expression.sum>`). Products with {py:func}`jm.prod() <jijmodeling.prod>` or {py:func}`Expression.prod() <jijmodeling.Expression.prod>`, and maximums and minimums with {py:func}`jm.max() <jijmodeling.max>` or {py:func}`jm.min() <jijmodeling.min>`, can be written in the same way.
:::

In the Decorator API, sums and products can be written intuitively as {external+python:ref}`comprehensions`.

The following example uses the Decorator API to write the sum of products of decision variables and placeholders:

In [10]:
@jm.Problem.define("Sum Example")
def sum_example(problem: jm.DecoratedProblem):
    N = problem.Length()
    a = problem.Float(shape=(N,))
    x = problem.BinaryVar(shape=(N,))
    problem += jm.sum(a[i] * x[i] for i in N)


sum_example

Problem(name="Sum Example", sense=MINIMIZE, objective=sum(stream(N).map(lambda (i: natural): a[i] * x[i])), constraints=[])

:::{admonition} Do Not Use Python's Built-in {py:func}`sum` Function
:class: caution

When writing a reduction with a Decorator API comprehension, use JijModeling's {py:func}`jm.sum() <jijmodeling.sum>`, {py:func}`jm.prod() <jijmodeling.prod>`, {py:func}`jm.max() <jijmodeling.max>`, or {py:func}`jm.min() <jijmodeling.min>`.
If you accidentally pass an expression such as `a[i] * x[i] for i in N` to Python's built-in {py:func}`sum`, Python tries to iterate over the JijModeling expression `N` at runtime, resulting in an error like the following:
:::

In [11]:
try:

    @jm.Problem.define("Wrong Sum Example")
    def wrong_sum_example(problem: jm.DecoratedProblem):
        N = problem.Length()
        a = problem.Float(shape=(N,))
        x = problem.BinaryVar(shape=(N,))
        # ERROR: using Python's built-in sum instead of jm.sum()
        problem += sum(a[i] * x[i] for i in N)
except Exception as e:
    print(e)

error[E-SE0000] JijModeling objects cannot be iterated at runtime.

Typical triggers:
  - comprehension syntax used outside the decorator API
  - Python's builtin `sum` used instead of `jijmodeling.sum`
  - a plain `for` loop, `list(...)`, or unpacking over a JijModeling object

Possible fix: move reductions into a function decorated with `@problem.update` or `@jijmodeling.Problem.define` and use JijModeling's own functions such as `jijmodeling.sum`.

Hint: You can read the description and possible fix at https://jij-inc-jijmodeling.readthedocs-hosted.com/en/stable/error_codes/error/E-SE0000.html


The same program can be written entirely with the Plain API using {py:func}`jijmodeling.map`, which will be explained in the next section:

In [12]:
sum_example_plain = jm.Problem("Sum Example (Plain)")
N = sum_example_plain.Length("N")
a = sum_example_plain.Float("a", shape=(N,))
x = sum_example_plain.BinaryVar("x", shape=(N,))
sum_example_plain += jm.sum(jm.map(lambda i: a[i] * x[i], N))

sum_example_plain

Problem(name="Sum Example (Plain)", sense=MINIMIZE, objective=sum(N.map(lambda (i: natural): a[i] * x[i])), constraints=[])

For a simple sum like this, you can also pass two arguments to {py:func}`jm.sum() <jijmodeling.sum>`: the domain and a function that returns the term to sum.

In [13]:
sum_example_plain_alt = jm.Problem("Sum Example (Plain, Alt)")
N = sum_example_plain_alt.Length("N")
a = sum_example_plain_alt.Float("a", shape=(N,))
x = sum_example_plain_alt.BinaryVar("x", shape=(N,))
sum_example_plain_alt += jm.sum(N, lambda i: a[i] * x[i])

sum_example_plain_alt

Problem(name="Sum Example (Plain, Alt)", sense=MINIMIZE, objective=sum(stream(N).map(lambda (i: natural): a[i] * x[i])), constraints=[])

:::{important}
This two-argument reduction form is supported only by {py:func}`jm.sum() <jijmodeling.sum>` and {py:func}`jm.prod() <jijmodeling.prod>`, not by {py:func}`jm.max() <jijmodeling.max>` or {py:func}`jm.min() <jijmodeling.min>`.

When using only the Plain API, expressions that traverse indices must be constructed with Python {external+python:ref}`lambda expressions <lambda>`.
:::

:::{tip}
When {py:func}`jm.sum() <jijmodeling.sum>` or {py:func}`jm.prod() <jijmodeling.prod>` is called as a single-argument function or method, it takes the sum or product of a stream. To sum the elements of `x`, you can simply write {py:func}`jm.sum(x) <jijmodeling.sum>` or {py:meth}`x.sum() <jijmodeling.Expression.sum>`. With the limited broadcasting described earlier, the example above can also be written as {py:func}`jm.sum(a * x) <jijmodeling.sum>`. The same applies when `x` is a two- or higher-dimensional array.
:::

Combining these reduction functions with `if` clauses in comprehensions allows more flexible reductions to be expressed.
For specific examples, see {doc}`../references/cheat_sheet`.

## Transforming Streams

So far, we have seen how to construct streams and consume them through reductions.
The following sections explain how to transform existing streams and combine multiple streams to construct new ones.

### Filtering Streams

{py:func}`~jijmodeling.filter` constructs a new stream containing only the elements of an existing stream that satisfy a given condition.

In [14]:
filter_problem = jm.Problem("Stream Filter Example")
N = filter_problem.Natural("N")
N.filter(lambda i: i % 2 == 0)

Expression(N.filter(lambda i: i % 2 == 0))

### Removing Duplicates from a Stream

As noted above, a stream may contain duplicate values.
When duplicates must be removed, use {py:func}`~jijmodeling.unique`. For each value that appears multiple times, only its first occurrence is retained, producing a stream of unique values.

In [15]:
problem = jm.Problem("Stream Uniquifization Example")
A = problem.Natural("x", ndim=1)
problem += A.unique().sum()  # Treat the array as a stream, remove duplicates, then sum

instance_data = {"x": [1, 3, 1, 2, 2, 1]}
instance = problem.eval(instance_data)
assert instance.objective == 6  # Only 1, 3, and 2 remain, so the sum is 6

### Mapping Streams

{py:func}`~jijmodeling.map`, corresponding to the Python standard library's {py:func}`~map`, constructs a new stream from the results of applying a function to the elements of an existing stream.

In [16]:
map_problem = jm.Problem("Stream Map Example")
N = map_problem.Natural("N")
x = map_problem.BinaryVar("x", shape=N)
map_problem += jm.sum(jm.stream(N).map(lambda i: x[i] ** 2))

map_problem

Problem(name="Stream Map Example", sense=MINIMIZE, objective=sum(stream(N).map(lambda (i: natural): x[i] ** 2)), constraints=[])

:::{admonition} Mapping Arrays and Dictionaries
:class: info

{py:func}`~jijmodeling.map` can also be called directly on arrays and dictionaries. In this case, the result is not a stream but a new array or dictionary with the same shape or key set.
In particular, because {py:func}`map <jijmodeling.map>` preserves shape and key-set information for these types, mapped elements can be accessed with the same indices as the original container.
As noted above, these types are automatically converted into streams, so stream operations behave the same way on a mapped container.
:::

### Flat-Mapping Streams

If the function passed to {py:func}`~jijmodeling.map` returns a stream, the result is a stream of streams.
Use {py:func}`jm.flat_map() <jijmodeling.flat_map>` (or its method form, {py:meth}`Expression.flat_map() <jijmodeling.Expression.flat_map>`) to flatten the mapped result by one level.
This makes it possible to traverse multiple indices without using Decorator API comprehensions.

In [17]:
flat_map_problem = jm.Problem("Stream FlatMap Example")
N = flat_map_problem.Natural("N")
M = flat_map_problem.Natural("M")

# A stream containing (i, 0), (i, 1), ..., (i, M-1) for each i
jm.stream(N).flat_map(lambda i: jm.map(lambda j: (i, j), M))

Expression(stream(N).flat_map(lambda i: M.map(lambda j: (i, j))))

### Cartesian Products of Streams

Use {py:func}`~jijmodeling.product` to take the Cartesian product of multiple streams.

In [18]:
product_problem = jm.Problem("Stream Product Example")
N = product_problem.Natural("N")
M = product_problem.Natural("M")
jm.product(N, M)

Expression(stream((N, M)))

Semantically, this is equivalent to traversing the elements of multiple streams with successive `for` clauses:

In [19]:
@product_problem.update
def _(problem: jm.DecoratedProblem):
    display(jm.stream((i, j) for i in N for j in M))

Expression(stream(stream(N.flat_map(lambda i: M.map(lambda j: (i, j)))).map(lambda (i, j): (i, j))))

Where a stream is expected, a tuple can be written directly to represent a Cartesian product, omitting {py:func}`~jijmodeling.product`.
Examples include the right-hand side of `in` in a Decorator API comprehension and the `domain=` keyword argument to {py:meth}`Problem.Constraint() <jijmodeling.Problem.Constraint>`.

In [20]:
@jm.Problem.define("Tuple Product Example")
def tuple_product_example(problem: jm.DecoratedProblem):
    N = problem.Length()
    M = problem.Length()
    Q = problem.Float(shape=(N, M))
    x = problem.BinaryVar(shape=(N, M))

    # A tuple represents the Cartesian product instead of jm.product
    problem += jm.sum(Q[i, j] * x[i, j] for (i, j) in (N, M))


tuple_product_example

Problem(name="Tuple Product Example", sense=MINIMIZE, objective=sum(stream((N, M)).map(lambda ((i, j): Tuple[natural, natural]): Q[i, j] * x[i, j])), constraints=[])

The same applies when passing `domain=` in the Plain API. Each component of the Cartesian product is passed to the `lambda` expression as a separate argument, in order.

In [21]:
tuple_domain_example = jm.Problem("Tuple Domain Example")
N = tuple_domain_example.Length("N")
M = tuple_domain_example.Length("M")
x = tuple_domain_example.BinaryVar("x", shape=(N, M))
tuple_domain_example += tuple_domain_example.Constraint(
    "bound", lambda i, j: x[i, j] <= 1, domain=(N, M)
)

tuple_domain_example

Problem(name="Tuple Domain Example", sense=MINIMIZE, objective=0, constraints={bound: [Constraint(name="bound", lambda (i, j): x[i, j] <= 1, domain=stream((N, M))),],})

## Logical Operations on Conditional Expressions and Streams

The conditions used in the `if` clauses and {py:func}`~jijmodeling.filter` functions above were simple, but conditions often need to combine logical expressions with “and” or “or.”
Because Python's logical operators `and`, `or`, and `not` cannot be overloaded, use the bitwise operators `&` (and), `|` (or), and `~` (not), or the functions {py:func}`jijmodeling.band` (and), {py:func}`jijmodeling.bor` (or), and {py:func}`jijmodeling.bnot` (not).

:::{admonition} Beware of Bitwise Operator Precedence
:class: caution

Unlike `and` and `or`, `&` and `|` have higher precedence than `==` and `!=`. For example, `a == b & c == d` is interpreted as `a == (b & c) == d`.
When using `&` or `|`, always enclose each comparison in parentheses, as in `(a >= b) & (c == d)`.
:::

Logical operations can also be applied to stream expressions: `|` represents a union and `&` represents an intersection.
Complementing a stream is not supported because the result could be infinite. Instead, {py:func}`jijmodeling.diff` computes the difference between two specific streams.